In [ ]:
# === Setup ===
import torch
import numpy as np
import matplotlib.pyplot as plt

from aind_behavior_vrforaging_analysis.sbi_ddm_analysis.simulator import PatchForagingDDM, create_prior
from features import extract_features 

# Reproducibility
torch.manual_seed(0)
np.random.seed(0)

# Initialize simulator and prior
simulator = PatchForagingDDM()
prior = create_prior()

# Helper to run simulation and extract features
def simulate_and_extract(theta, window_sites=100):
    """
    Runs one simulation for a given theta and extracts its features.
    Returns (window, features, summary_stats)
    """
    theta_tensor = torch.tensor(theta, dtype=torch.float32)
    window = simulator.simulate_with_random_walk(
        theta_mean=theta_tensor,
        window_sites=window_sites,
        random_walk_sigma=0.0
    )
    features = extract_features(window)
    summary_stats = features[-4:].detach().numpy()
    # summary_stats = features[:4].detach().numpy()
    return window, features, summary_stats


In [ ]:
# === Sweep Setup with Color Gradient ===
theta_base = np.array([0.5, 0.5, 0.5])
theta_labels = ["drift_rate", "reward_bump", "failure_bump"]
summary_names = [
    "max_time",
    "mean_reward_per_window",
    "mean_stopped_per_window",
    "frac_failures_per_window"
]

window_sites = 100
n_repeats = 10  # number of runs per theta for CI

# Define 5 gradient levels for the color-varying parameter
gradient_values = np.linspace(0.01, 2.0, 5)
x_values = np.linspace(0.01, 2.0, 10)  # values along x-axis

# Prepare storage for results: mean and sem
# Structure: results[param_x][param_color_value] = mean/sem arrays
results_mean = {label: {} for label in theta_labels}
results_sem = {label: {} for label in theta_labels}


In [ ]:
for i, param_x in enumerate(theta_labels):
    other_params = [p for p in theta_labels if p != param_x]
    fixed_param = other_params[1]  # keep the third parameter fixed at 0.5

    for val_color in gradient_values:
        mean_list, sem_list = [], []

        for x_val in x_values:
            theta = theta_base.copy()
            theta[i] = x_val  # x-axis parameter
            # Vary the "color" parameter
            color_idx = theta_labels.index(other_params[0])
            theta[color_idx] = val_color
            # Fixed parameter remains base value (0.5)

            runs = []
            for _ in range(n_repeats):
                _, _, summary = simulate_and_extract(theta, window_sites)
                runs.append(summary)
            runs = np.vstack(runs)
            mean_list.append(runs.mean(axis=0))
            sem_list.append(runs.std(axis=0, ddof=1)/np.sqrt(n_repeats))

        # Convert val_color to string to avoid floating point key issues
        results_mean[param_x][str(val_color)] = np.vstack(mean_list)
        results_sem[param_x][str(val_color)] = np.vstack(sem_list)


In [ ]:
for i, param_x in enumerate(theta_labels):
    other_params = [p for p in theta_labels if p != param_x]
    fixed_param = other_params[1]

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))  # 1 row, 5 columns
    axes = axes.flatten()
    fig.suptitle(f"Effect of {param_x} with gradient in {other_params[0]}", fontsize=16)
    axes = axes.flatten()

    cmap = plt.cm.get_cmap("viridis", len(gradient_values))

    for j, feature_name in enumerate(summary_names):
        ax = axes[j]

        for idx, val_color in enumerate(gradient_values):
            key = str(val_color)  # convert to string to match storage
            mean_vals = results_mean[param_x][key][:, j]
            sem_vals = results_sem[param_x][key][:, j]

            ax.plot(x_values, mean_vals, color=cmap(idx), marker="o", label=f"{other_params[0]}={val_color:.2f}")
            ax.fill_between(
                x_values,
                mean_vals - 1.96 * sem_vals,
                mean_vals + 1.96 * sem_vals,
                color=cmap(idx),
                alpha=0.2,
                linewidth=0
            )

        ax.set_title(feature_name, fontsize=10)
        ax.set_xlabel(param_x)
        ax.set_ylabel("Value")
        ax.grid(True, linestyle="--", alpha=0.6)

    for k in range(len(summary_names), len(axes)):
        axes[k].axis("off")

    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper right', title=f"{other_params[0]} values")

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()
